In [ ]:
import os
import sys

# Dynamic root path scoping
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.data_loader import load_insurance_data
from src.hypothesis_tests import test_numerical_margin, test_categorical_frequency

# Load the cleaned dataset state from task 2
df = load_insurance_data('../data/cleaned_insurance_data.csv')

# Explicitly cast structural target metrics
df['HasClaimed'] = (df['TotalClaims'] > 0).astype(int)
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print("Data loaded and structured successfully!")

In [ ]:
gender_res = test_categorical_frequency(df, 'Gender', 'HasClaimed')
print(f"=== Gender Chi-Square Test ===")
print(f"Statistic: {gender_res['statistic']:.4f}")
print(f"P-Value: {gender_res['p_value']:.4e}")
print(f"Reject Null Hypothesis? {gender_res['reject_h0']}\n")

# Visual Verification
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Gender', hue='HasClaimed', palette='muted')
plt.title('Claim Distribution by Gender Profile')
plt.show()

In [ ]:
# Select top 5 ZipCodes by concentration to ensure structural sampling stability
top_zipcodes = df['ZipCode'].value_counts().nlargest(5).index
filtered_zip_df = df[df['ZipCode'].isin(top_zipcodes)]

zip_res = test_categorical_frequency(filtered_zip_df, 'ZipCode', 'HasClaimed')
print(f"=== ZipCode Chi-Square Test ===")
print(f"Statistic: {zip_res['statistic']:.4f}")
print(f"P-Value: {zip_res['p_value']:.4e}")
print(f"Reject Null Hypothesis? {zip_res['reject_h0']}\n")

In [ ]:
gauteng_margin = df[df['Province'] == 'Gauteng']['Margin']
wc_margin = df[df['Province'] == 'Western Cape']['Margin']

margin_res = test_numerical_margin(gauteng_margin, wc_margin)
print(f"=== Province Margin Two-Sample T-Test ===")
print(f"T-Statistic: {margin_res['statistic']:.4f}")
print(f"P-Value: {margin_res['p_value']:.4e}")
print(f"Reject Null Hypothesis? {margin_res['reject_h0']}\n")

# Visual Verification
plt.figure(figsize=(8, 4))
sns.kdeplot(gauteng_margin, shade=True, color="teal", label="Gauteng Margin", clip=(-5000, 20000))
sns.kdeplot(wc_margin, shade=True, color="coral", label="Western Cape Margin", clip=(-5000, 20000))
plt.title('Underwriting Margin Distribution Profiles')
plt.xlabel('Margin value (ZAR)')
plt.legend()
plt.show()